In [ ]:
import multiprocessing as mp
import os
import time

In [ ]:
DOCUMENTS = [
    "distributed systems are fun",
    "map reduce is a programming model",
    "the master splits input into map tasks",
    "reduce tasks sum the counts from map tasks",
    "fun fun fun distributed computing",
]

MAP_TIMEOUT = 2.0

In [ ]:
def flaky_map_task(doc_id, text, attempt):
    pid = os.getpid()
    print(f"[MAP worker pid={pid}] attempt #{attempt} on doc {doc_id}: {text!r}")

    if doc_id == 1 and attempt == 1:
        print(f"[MAP worker pid={pid}] CRASHING on doc {doc_id}")
        raise RuntimeError("simulated worker crash")

    if doc_id == 3:
        time.sleep(3.0)

    pairs = [(word, 1) for word in text.split()]
    return pairs

In [ ]:
def run_map_with_retry(pool, doc_id, text, max_attempts=3):
    for attempt in range(1, max_attempts + 1):
        async_result = pool.apply_async(
            flaky_map_task,
            (doc_id, text, attempt)
        )

        try:
            return async_result.get(timeout=MAP_TIMEOUT)

        except mp.TimeoutError:
            print(
                f"[MASTER] doc {doc_id} attempt #{attempt} is a STRAGGLER "
                f"(> {MAP_TIMEOUT}s)"
            )

            backup_result = pool.apply_async(
                flaky_map_task,
                (doc_id, text, attempt)
            )

            return backup_result.get()

        except Exception as e:
            print(
                f"[MASTER] doc {doc_id} attempt #{attempt} CRASHED ({e})"
            )

    raise RuntimeError(f"doc {doc_id} failed after {max_attempts} attempts")

In [ ]:
def main():
    print("[MASTER] Splitting input into", len(DOCUMENTS), "map tasks")

    with mp.Pool(processes=4) as pool:
        all_pairs = []

        for doc_id, text in enumerate(DOCUMENTS):
            pairs = run_map_with_retry(pool, doc_id, text)
            all_pairs.extend(pairs)

    counts = {}

    for word, c in all_pairs:
        counts[word] = counts.get(word, 0) + c

    print("\n[MASTER] All map tasks eventually succeeded.")
    print("[MASTER] FINAL WORD COUNTS:")

    for word, count in sorted(
        counts.items(),
        key=lambda kv: -kv[1]
    ):
        print(f"  {word:12s} {count}")

In [ ]:
if __name__ == "__main__":
    main()

[MASTER] Splitting input into 5 map tasks
[MAP worker pid=30530] attempt #1 on doc 0: 'distributed systems are fun'
[MAP worker pid=30533] attempt #1 on doc 1: 'map reduce is a programming model'
[MAP worker pid=30533] CRASHING on doc 1
[MAP worker pid=30532] attempt #2 on doc 1: 'map reduce is a programming model'
[MAP worker pid=30531] attempt #1 on doc 2: 'the master splits input into map tasks'
[MAP worker pid=30532] attempt #1 on doc 3: 'reduce tasks sum the counts from map tasks'
[MASTER] doc 1 attempt #1 CRASHED (simulated worker crash)
[MAP worker pid=30533] attempt #1 on doc 3: 'reduce tasks sum the counts from map tasks'
[MASTER] doc 3 attempt #1 is a STRAGGLER (> 2.0s)
[MAP worker pid=30530] attempt #1 on doc 4: 'fun fun fun distributed computing'

[MASTER] All map tasks eventually succeeded.
[MASTER] FINAL WORD COUNTS:
  fun          4
  map          3
  tasks        3
  distributed  2
  reduce       2
  the          2
  systems      1
  are          1
  is           1
  a 